# OtakuScan Phase 3: One-Click Google Colab GPU Workflow & VLM Pilot

This notebook provides an automated, one-click setup and execution workflow for **OtakuScan Phase 3** in Google Colab Premium.

### Initial One-Time Requirement:
- Upload the local `dataset/` archive to Google Drive as:
  `MyDrive/otakuscan-dataset/otakuscan-dataset.tar.gz`

### Execution Instructions:
1. Open this notebook directly from GitHub in Google Colab.
2. Select a GPU runtime: **Runtime > Change runtime type > T4 GPU** (or A100).
3. Run all cells sequentially (**Runtime > Run all**).

## 1. GPU Detection & Hardware Audit
Verify GPU accelerator availability and available VRAM.

In [ ]:
!nvidia-smi

import sys
import torch

print("=" * 65)
print("COLAB GPU ACCELERATOR AUDIT")
print("=" * 65)
print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Active GPU:      {gpu_name}")
    print(f"Dedicated VRAM:  {vram_gb:.2f} GB")
    print("STATUS: GPU hardware verified and ready.")
else:
    print("ERROR: No CUDA GPU detected!")
    print("Please select: Runtime > Change runtime type > T4 GPU before continuing.")
print("=" * 65)

## 2. Install Dependencies
Install required dependencies for RapidOCR, transformers vision-language processing, and acceleration.

In [ ]:
!pip install -q rapidocr-onnxruntime pillow numpy scipy "transformers>=4.45.0" accelerate qwen-vl-utils

## 3. One-Click Workspace & Dataset Setup (Idempotent)
Automates Drive mounting, repository location/cloning, and idempotent dataset extraction.
Configures `project_root` and `dataset_root` independently.

In [ ]:
import os
import sys
import tarfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = "https://github.com/neolakshay/otakuscan.git"
EXPECTED_ARCHIVE_REL = "otakuscan-dataset/otakuscan-dataset.tar.gz"
EXTRACT_TARGET = Path("/content/otakuscan-data/dataset")

def check_valid_dataset(p: Path) -> Path:
    if not p.is_dir():
        return None
    if (p / "sequences.json").is_file() and (p / "development" / "labels.jsonl").is_file():
        return p
    nested = p / "dataset"
    if nested.is_dir() and (nested / "sequences.json").is_file() and (nested / "development" / "labels.jsonl").is_file():
        return nested
    return None

print("=" * 65)
print("ONE-CLICK WORKSPACE & DATASET SETUP")
print("=" * 65)

if IN_COLAB:
    # 1. Mount Google Drive
    drive_root = Path("/content/drive")
    my_drive = drive_root / "MyDrive"
    if not my_drive.is_dir():
        print("Mounting Google Drive...")
        from google.colab import drive
        drive.mount(str(drive_root))
    else:
        print("Google Drive already mounted.")

    # 2. Locate or clone project repository
    colab_repo = Path("/content/otakuscan")
    if (colab_repo / "splits" / "dev_splits.json").is_file():
        project_root = colab_repo
        print(f"Repository already present at: {project_root}")
    else:
        print(f"Cloning project repository from {REPO_URL} into {colab_repo}...")
        !git clone {REPO_URL} /content/otakuscan
        project_root = colab_repo

    # 3. Locate dataset archive in Google Drive
    primary_archive = my_drive / EXPECTED_ARCHIVE_REL
    fallback_archive = my_drive / "otakuscan-dataset.tar.gz"

    existing_dataset = check_valid_dataset(EXTRACT_TARGET)
    if existing_dataset is not None:
        dataset_root = existing_dataset
        print(f"Existing valid dataset found at: {dataset_root} (reusing without re-extracting)")
    else:
        if primary_archive.is_file():
            archive_path = primary_archive
        elif fallback_archive.is_file():
            archive_path = fallback_archive
        else:
            raise FileNotFoundError(
                f"Dataset archive not found in Google Drive!\n"
                f"Checked paths:\n"
                f"  1. {primary_archive}\n"
                f"  2. {fallback_archive}\n"
                f"Please upload your local 'dataset/' archive to Google Drive as:\n"
                f"  MyDrive/otakuscan-dataset/otakuscan-dataset.tar.gz"
            )

        print(f"Extracting dataset archive from {archive_path} to {EXTRACT_TARGET}...")
        EXTRACT_TARGET.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive_path, "r:*") as tar:
            if hasattr(tarfile, 'data_filter'):
                tar.extractall(path=EXTRACT_TARGET, filter='data')
            else:
                tar.extractall(path=EXTRACT_TARGET)

        dataset_root = check_valid_dataset(EXTRACT_TARGET)
        if dataset_root is None:
            raise FileNotFoundError(f"Extracted archive at {EXTRACT_TARGET} did not contain sequences.json!")
        print(f"Dataset extracted and verified successfully at: {dataset_root}")

else:
    # Local machine execution fallback
    project_root = Path.cwd().resolve()
    while not (project_root / 'splits' / 'dev_splits.json').is_file() and project_root.parent != project_root:
        project_root = project_root.parent
    dataset_root = check_valid_dataset(project_root / 'dataset')
    if dataset_root is None:
        raise FileNotFoundError(f"Local dataset not found at {project_root / 'dataset'}")

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Validate setup
from src.data.loader import OtakuDatasetLoader
loader = OtakuDatasetLoader(dataset_root)
seqs = loader.load_sequences()
labels = loader.load_labels('development')

print("-" * 65)
print(f"project_root:   {project_root}")
print(f"dataset_root:   {dataset_root}")
print(f"Sequences:      {len(seqs)} records loaded")
print(f"Dev Labels:     {len(labels)} sequences loaded")
print("STATUS: One-click workspace setup completed successfully!")
print("=" * 65)

## 4. Strict Leakage Prevention & Split Verification
Verify that training uses strictly Fold 0 training sequences (64 sequences), and that validation sequences (16 sequences) and test sequences (15 sequences) remain strictly isolated.

In [ ]:
import json

with open(project_root / 'splits' / 'dev_splits.json', 'r', encoding='utf-8') as f:
    splits_data = json.load(f)

train_ids = set(splits_data['kfold_5'][0]['train'])
val_ids = set(splits_data['kfold_5'][0]['val'])

print(f"Fold 0 Training Sequences:   {len(train_ids)}")
print(f"Fold 0 Validation Sequences: {len(val_ids)}")

# Enforce zero sequence leakage
overlap = train_ids.intersection(val_ids)
assert len(overlap) == 0, f"CRITICAL FAILURE: Sequence leakage detected! Overlapping IDs: {overlap}"
print("PASSED: Zero sequence leakage between training and validation sets.")

## 5. Verify & Retrain Adapted Classifiers
Run training and hyperparameter sensitivity grid search on Fold 0 training sequences using `dataset_root`.

In [ ]:
from colab.train_classifier_colab import run_colab_training

train_summary = run_colab_training(
    dataset_root=dataset_root,
    splits_file=project_root / 'splits' / 'dev_splits.json',
    output_dir=project_root / 'models',
    run_grid_search=True,
)

print("\n--- Verification Summary ---")
print(f"Training samples: {train_summary['train_samples']}")
print(f"Weights saved to: {train_summary['filter_weights_path']}")
print(f"Grid search configurations evaluated: {len(train_summary['grid_results'])}")

## 6. Vision-Language Model (VLM) Pilot Experiment
Execute the controlled VLM pilot benchmark (`Qwen/Qwen2-VL-2B-Instruct`) against RapidOCR on 3 permitted Fold 0 training pages using `dataset_root`.

In [ ]:
from colab.vlm_manga_transcription_experiment import run_pilot_experiment

pilot_output_file = project_root / 'logs' / 'colab_vlm_pilot_results.json'

pilot_results = run_pilot_experiment(
    project_root=project_root,
    dataset_root=dataset_root,
    model_id="Qwen/Qwen2-VL-2B-Instruct",
    n_pages=3,
    output_json=pilot_output_file,
    require_gpu=True,
)

## 7. Comparative Audit & Results Inspection
Inspect side-by-side performance (latency, VRAM, token F1) and download the results log.

In [ ]:
print("=" * 78)
print(f"{'Page':<12} {'Engine':<18} {'Latency':<12} {'Peak VRAM':<14} {'Token F1':<12}")
print("=" * 78)

for p in pilot_results['per_page_results']:
    p_name = p['page_name']
    # RapidOCR
    ocr_lat = f"{p['rapidocr']['latency_seconds']}s"
    ocr_f1 = f"{p['rapidocr']['token_match']['token_f1']:.4f}"
    print(f"{p_name:<12} {'RapidOCR':<18} {ocr_lat:<12} {'< 65 MB':<14} {ocr_f1:<12}")
    
    # VLM
    vlm_info = p['vlm']
    if vlm_info.get('status') == 'success':
        vlm_lat = f"{vlm_info['latency_seconds']}s"
        vlm_vram = f"{vlm_info['peak_vram_mb']:.1f} MB"
        vlm_f1 = f"{vlm_info['token_match']['token_f1']:.4f}"
        print(f"{p_name:<12} {'Qwen2-VL-2B':<18} {vlm_lat:<12} {vlm_vram:<14} {vlm_f1:<12}")
    else:
        print(f"{p_name:<12} {'Qwen2-VL-2B':<18} {'N/A':<12} {'N/A':<14} {vlm_info.get('status', 'skipped'):<12}")
    print("-" * 78)

print("\nPILOT LOG SAVED AT:", pilot_output_file)
print("Please download logs/colab_vlm_pilot_results.json and share the recorded metrics.")